In [20]:
from crewai.llm import LLM
import os

azure_llm = LLM(
    model="azure/gpt-4o-mini",
    api_key= os.getenv("AZURE_OPENAI_API_KEY"),
    api_base= os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    is_litellm=True
)

In [25]:
from crewai.flow.flow import Flow, listen, start
import asyncio
from pydantic import BaseModel

class CityInfo(BaseModel):
    city: str = ""

class CityFacts(Flow[CityInfo]):
    info = CityInfo()
    
    @start()
    def generate_city(self):
        print("Starting flow: generating city...")
        self.info.city = azure_llm.call("Suggest a single city in name in the world, dont include any other message only the city name.")
    
    @listen(generate_city)
    def create_guide(self, city: str):
        return azure_llm.call(f"Provide a fun fact about the {self.info.city}, dont include any other additional message.")

# Async execution for notebook compatibility
await CityFacts().kickoff_async()

Flow started with ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
┌────────────────────────────── Flow Execution ───────────────────────────────┐
│                                                                             │
│  Starting Flow Execution                                                    │
│  Name: CityFacts                                                            │
│  ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘


Flow started with ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5



Starting flow: generating city...
✅ Flow Finished: CityFacts
├── ✨ Created
✅ Flow Finished: CityFacts
├── ✨ Created
✅ Flow Finished: CityFacts
├── ✨ Created
✅ Flow Finished: CityFacts
├── ✨ Created
✅ Flow Finished: CityFacts
├── ✨ Created
✅ Flow Finished: CityFacts
├── ✨ Created
✅ Flow Finished: CityFacts
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
🌊 Flow: CityFacts
ID: b7fa6137-1ed4-4e6d-a957-a8cda1b740a5
├── ✨ Created
c:\Ai Support Companion\CrewAI\.venv\Lib\site-packages\pydantic\main.py:463: 
  PydanticSerial

'Kyoto is home to over 1,600 Buddhist temples and more than 400 Shinto shrines, making it one of the most culturally rich cities in Japan.'

# **State Management & Dynamic Routing**

In [ ]:
'''
Two Approaches to State Management
CrewAI offers two ways to manage state in your flows:
- Unstructured State - Using dictionary-like objects for flexibility

- Structured State - Using Pydantic models for type safety and validation
class AppState(BaseModel):
    Age: int = 0
    name: str = ""
'''

In [ ]:
from crewai.flow.flow import Flow, listen, start, router, or_
from crewai.flow.persistence import persist
import asyncio
from pydantic import BaseModel, Field
import random

class AppState(BaseModel):
    value: int = Field(default=0, description="An integer value representing some state.")
    name: str = Field(default="", description="A name associated with the state.")
    count: int = Field(default=0, description="A counter for tracking purposes.")

class StructuredStateFlow(Flow[AppState]):
    def __init__(self, name="unknown"):
        super().__init__()
        self.name = name

    @start()
    def initialize_state(self):
        self.state.value = random.randint(1, 2)
        self.state.name = self.name
        self.state.count += 1
        print(f"Initialized state value to {self.state.value} for {self.state.name}.")
        return f"Value is {self.state.value}"
    
    @router(initialize_state)
    def routing_logic(self):
        self.state.count += 1
        print(f"DEBUG: routing_logic called with state value {self.state.value}")
        return "Path A" if self.state.value == 1 else "Path B"

    @listen("Path A")
    def path_A(self):
        self.state.count += 1
        return f"Executed Path A for {self.state.name}."
    
    @listen("Path B")
    def path_B(self):
        self.state.count += 1
        return f"Executed Path B for {self.state.name}."
    
    @listen(or_("Path A", "Path B"))
    def finalize(self, message):
        print(f"{message}")

# Async execution for notebook compatibility - pass name as constructor parameter
await StructuredStateFlow(name="Alice").kickoff_async()
# await StructuredStateFlow(name="Bob").kickoff_async()

Flow started with ID: a2f845b2-eace-4a66-a746-3b8ae85cc05c
✅ Flow Finished: Alice
├── ✨ Created
✅ Flow Finished: Alice
├── ✨ Created
└── ✅ Initialization Complete
┌────────────────────────────── Flow Execution ───────────────────────────────┐
│                                                                             │
│  Starting Flow Execution                                                    │
│  Name: Alice                                                                │
│  ID: a2f845b2-eace-4a66-a746-3b8ae85cc05c                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

Flow started with ID: a2f845b2-eace-4a66-a746-3b8ae85cc05c
✅ Flow Finished: Alice
├── ✨ Created
└── ✅ Initialization

'Path Selection Has Been Made for Alice.'

In [59]:
import random
for i in range(10):
    print(random.randint(1, 2), end=" - ")

1 - 2 - 1 - 2 - 1 - 1 - 2 - 1 - 2 - 1 - 